In [ ]:
## This is an exploration of the PCN 
## Setting up the enviornment
import os
os.environ["JAX_PLATFORMS"]= 'cpu'

import pcn
print(sys.executable)

import numpy as np
import jax.numpy as jnp
import optax
import pcn

In [ ]:
## Exploration: Constructing a toy multimodal vowel/consonant classifier

SEED = 9
N_TRAIN, N_TEST = 800,200 # examples to make
BATCH = 40  # how many examples the network looks at in one go


SR = 16000 # sample rate in hertz
CLIP = 4096 # length of one sound clip in samples
N_MELS = 16 # how many frequency bands that the ear produces
N_VIDEO = 4 # number of mouth measurements per example, i.e. opening, width, rounding, jaw



In [ ]:
front = pcn.PCNetwork (seed = SEED) # network to simulate ear

with front:
    ear = pcn.AuditoryInput(n_samples = CLIP, sr = SR, n_mels = N_MELS, label = 'ear')

AUDIO_DIM = ear.dim # numbers per clip after encoding

print('raw',ear.raw_shape,'-> mel features', ear.feature_shape,'-> flat dim',AUDIO_DIM)

In [ ]:
# compose sound generators

def vowel_wave(rng,t):
    # This is once voiced lip, with harmonics on a pitch, shaped by two peaks
    # rng is a numpy random generator, with t denoting time axis in seconds
    f0 = rng.uniform(180, 300) # higher pitch for child-direct speech

    f1,f2 = rng.uniform(300,800), rng.uniform(1000,2500) # Constructing formant centers
    wav = np.zeros_like(t)

    for k in range(1,20): # 19 harmonics of the pitch
        fk = k * f0
        gain = np.exp(-(((fk - f1) / 200) ** 2) + 0.6 * np.exp(-((fk - f2) / 300) ** 2))
        wav += gain * np.sin(2 * np.pi * fk *t + rng.uniform(0,2 * np.pi))

    return wav

        
                      
# compose consonants

def consonant_wave ( rng, t):
    # same inputs and ouput shape as vowel_wave

    noise = rng.standard_normal(len(t))
    noise = np.diff(noise, prepend = 0) #pushing energy t hihger bands

    env = np.exp(-t/rng.uniform(0.03,0.1)) # shape onset with fast decay

    return noise * env


# checking 
t = np.arange(CLIP)/ SR
v,c = vowel_wave ( rng := np.random.default_rng(SEED),t), consonant_wave(rng,t)
print('vowel clip', v.shape, 'consonant clip',c.shape)

In [ ]:
# Assembling the data set together

def make_clip (is_vowel, rng,t, audio_noise):#  one sound clip,normalized to the same loudness

    w = vowel_wave (rng, t) if is_vowel else consonant_wave(rng,t)

    w = w/(np.abs(w).max() + 1e-6)
    return w + rng.normal ( 0, audio_noise, len(t))

def make_mouth(is_vowel,rng, video_noise):
    
    # one set of mouth features: open for vowel, closed otherwise
    if is_vowel:
        mouth = np.array([1.0,0.7, rng.uniform(0,1),0.8])

    else:
        mouth = np.array([0.1,0.4,0.2,0.1])

    return mouth+ rng.normal(0, video_noise, N_VIDEO)
    
def simulate (n, rng, audio_noise = 0.05, video_noise = 0.3):# with n labelled examples as a dictionary of arrays, audio, video, label 
    y = rng.integers (0,2, size = n)
     # truth : 0 consonant, 1 vowel
    t = np.arange(CLIP) / SR
    waves = np.stack([make_clip(v, rng, t, audio_noise) for v in y]).astype(np.float32)
    video = np.stack([make_mouth(v, rng,video_noise) for v in y]).astype(np.float32)
    audio = np.asarray(ear.encode(jnp.asarray(waves))) # putting all clips through the ear
    label wha= np.eye(2, dtype = np.float32)[y] # one-hot encoding the truth about whether each example is a consonant or a vowel
    return {"audio":audio,"video": video, "label": label, "label_idx": y,"wave":waves}
     
     

In [ ]:
# Test if the set up will work

rng = np.random.default_rng(SEED)
d = simulate (5,rng)
print({k:v.shape for k, v in d.items()})
print('labels:',d['label_idx'])


In [ ]:
# Scaling and batching the data, further data preparation
def standardize (d, mu,sd): #rescale audio features to mean 0
    return {**d,'audio': ((d["audio"] - mu) / sd).astype(np.float32)}

def batches (d, batch): #cut a dataset dictionary into a list of smaller dictionary , one per batch
    n = len(d['label_idx'])
    return [{k: v[i:i + batch] for k, v in d.items()} for i in range(0, n, batch)]

rng = np.random.default_rng(SEED)
train_raw = simulate ( N_TRAIN,rng)
test_raw = simulate(N_TEST,rng)

mu = train_raw['audio'].mean(0)
sd = train_raw['audio'].std(0) + 1e-6
train = batches(standardize(train_raw, mu,sd),BATCH)
test = batches(standardize(test_raw, mu,sd),BATCH)

#Inspect results
print(len(train),'train batches,',len(test),'test batches')
print('audio range after scaling:',train[0]['audio'].min().round(1),'to',train[0]['audio'].max().round(1))





In [ ]:
## Constructing the network

HIDDEN = 32  #32 dimensions of hidden layer

def sensory_layers(): 
    #input layers, one per modality

    aud = pcn.Layer(dim = AUDIO_DIM, activation = pcn.Direct(), label = 'audio')

    vid = pcn. Layer(dim = N_VIDEO, activation = pcn.Direct(), label = 'video')

    return aud, vid
    

In [ ]:
## Construct the hidden layers
def hidden_layers():
    # one hidden layer per modality , they connect during inference
    
    #audio
    h_aud = pcn.Layer(dim = HIDDEN, activation = pcn.LeakyRelu(), label = 'hidden_aud')
    #visual
    h_vid = pcn.Layer(dim = HIDDEN, activation = pcn.LeakyRelu(), label = 'hidden_vid')
    return h_aud, h_vid
    

In [ ]:
## Construct the label layer, which is the two-unit softmax
def label_layer():
    return pcn.Layer(dim = 2, activation = pcn.Softmax(temperature = 0.5) , label = 'label')

In [ ]:
## Connect the layers together

def connect(aud, vid, h_aud, h_vid, label):
    ##Predict (source, and target)
    pcn.Predict(aud,h_aud) # audio predicts its hidden layer
    pcn.Predict(vid,h_vid) # video predicts its hidden layer
    pcn.Predict([h_aud, h_vid], label) # both hidden layers jointly predict the label
    

In [ ]:
## Assemble the model
def build_network(seed = SEED):
    
    net = pcn.PCNetwork(seed = SEED)
    net.config(use_bias = True, learn_precision_weights = False, learn_precision_bias = False)
    with net: # layers register with the net that is open

        aud, vid = sensory_layers()
        h_aud, h_vid = hidden_layers()
        label = label_layer()
        connect(aud, vid, h_aud, h_vid, label)

    net. build() # compile the layout
    return net, {'audio': aud, 'video':vid,'label':label}

net, L = build_network()
print('layers:', [l.label for l in net.structure.layers])
print('connections:', len(net.structure.predict_conns))

        
        